In [1]:
# For processing the timeseries
import pandas as pd, os, datetime
import numpy as np

# For plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/nem_generation'

In [3]:
sdate, edate = '2018-12-15','2019-02-15'

In [4]:
gen_details = pd.read_csv(f"{nmap_path}/gen_details.csv")

In [5]:
hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")

In [6]:
def process_group(grp, gen_fpath, hw_tseries, start_date=sdate, end_date=edate):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    gen_locs = gen_fpath + '/' + grp['DUID'] + ".csv"
    dfs = [pd.read_csv(fp,dtype='object') for fp in gen_locs if os.path.exists(fp)]

    if not dfs:
        return None

    # This is in case of accidental mid-file headers
    dfs = pd.concat(dfs, ignore_index=True)
    header_row = dfs.columns.tolist()
    dfs = dfs[~dfs.apply(lambda row: list(row) == header_row, axis=1)]

    # This is to correct the column types after removing header rows
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs['TOTALCLEARED'] = dfs['TOTALCLEARED'].astype(float)
    dfs['TOTALMWh'] = dfs['TOTALMWh'].astype(float)
    dfs['AGCSTATUS'] = pd.to_numeric(dfs['AGCSTATUS'], errors='coerce').fillna(0).astype(int)
    
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    agg_func = {'TOTALMWh':'sum','TOTALCLEARED':'sum','AGCSTATUS':'min'}
    dfs = dfs.groupby(['DUID', pd.Grouper(freq='1h')]).agg(agg_func)

    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index(['time']).sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate]
    hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

    merged = pd.merge_asof(
        dfs.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )

    return merged

In [7]:
def filter_hw_tseries(hw, df, sdate, edate):
    hw_tseries = hw[hw['DUID'].isin(df['DUID'])]
    
    hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])
    hw_tseries = hw_tseries.set_index('time').sort_index()
    hw_tseries = hw_tseries.loc[sdate:edate].reset_index()

    pivot = hw_tseries.pivot_table(index='time', columns='DUID', values='EHF_flag')
    
    # Considers a non-heatwave period of <10% of generators are in a heatwave
    all_zero = (pivot == 0).all(axis=1)

    valid_times = all_zero[all_zero].index.to_series()

    time_diffs = valid_times.diff()
    gap = (time_diffs != time_diffs.min()).cumsum()

    result = valid_times.groupby(gap).agg(['min', 'max']).rename(columns={"min": "Start Date", "max": "End Date"})
    result["EHF_flag"] = 0.0
    result["Note"] = "All DUIDs"

    return result.reset_index(drop=True)

In [8]:
groups = gen_details.groupby('region', as_index = False)
df = groups.get_group('NSW1')

In [9]:
# # This is to select all periods where <10% of generators are in a  heatwave
# hw_summ = filter_hw_tseries(hw_tseries,df,sdate,edate)
# hw_summ

06/01/2019-11/01/2019 BEFORE
12/01/2019-31/01/2019 DURING 
08/02/2019-15/02/2019 AFTER

In [10]:
# This retrieves only the time surrounding the heatwave
df = process_group(df, gen_fpath, hw_tseries, sdate, edate)
df = df.merge(gen_details[['DUID','fuel_source_primary', 'reg_cap_mw']], left_on='DUID', right_on='DUID', how='left')

In [11]:
df['reg_cap_mw'] = df['reg_cap_mw'].astype('float')
df['cap_norm'] = df['TOTALMWh']/df['reg_cap_mw']

In [12]:
demand = pd.read_csv('/scratch/ng72/ms5578/time_series/state_demand.csv')
demand['time'] = pd.to_datetime(demand['time'])
demand = demand.set_index('time').sort_index()
demand = demand.loc[sdate:edate]
demand = demand[demand["REGIONID"] ==  'VIC1'].drop('REGIONID',axis=1)
demand = demand.resample('h').sum()

In [13]:
def plot_multivars(df, title='Time Series Plot',lines=['TOTALMWh'], highlight = False):
    fig = go.Figure()

    for line in lines:
        fig.add_trace(go.Scatter(
            x=df['time'],
            y=df[line],
            mode='lines',
            name=str(line)
        ))
        
        if highlight == True:
            # Highlight the EHF flag
            fig = highLights(
                df=df,             # Only this group's data
                fig=fig,
                variable='EHF_flag',  # Column to check
                level=0,              # Threshold
                mode='above',         # or 'below'
                fillcolor='rgba(255,0,0,0.1)',  # Semi-transparent red
                layer='below'
            )

    fig.update_layout(
        title=title,
        xaxis_title='Time',
        yaxis_title='Total MWh',
        template='plotly_white',
        legend_title='Legend'
    )

    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(count=1,
                         label="1m",
                         step="month",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )

    return fig
    

In [14]:
def plot_agg_group(
    grouped_df, group_col, title='Time Series Plot', y='TOTALMWh',
    highlight=False, highlight_mode='union'
):
    fig = go.Figure()

    for group_name, group in grouped_df.groupby(group_col):
        fig.add_trace(go.Scatter(
            x=group['time'],
            y=group[y],
            mode='lines',
            name=str(group_name)
        ))

    fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0='2019-01-13',
            y0=0,
            x1='2019-02-09',
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(0,0,0,0.1)",
            layer='below')
    
    fig.add_shape(
                type="rect",
                xref="x",
                yref="paper",
                x0='2019-01-24',
                y0=0,
                x1='2019-01-26',
                y1=1,
                line=dict(color="rgba(0,0,0,0)", width=3),
                fillcolor="rgba(255,0,0,0.2)",
                layer='below')

    fig.update_layout(
        title=title,
        xaxis_title='Time',
        yaxis_title='Total MWh',
        template='plotly_white',
        legend_title=group_col
    )

    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(count=1,
                         label="1m",
                         step="month",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )

    return fig

In [15]:
def highLights(df, fig, variable, level, mode, fillcolor, layer):
    """
    Set a specified color as background for given
    levels of a specified variable using a shape.
    
    Keyword arguments:
    ==================
    fig -- plotly figure
    variable -- column name in a pandas dataframe
    level -- int or float
    mode -- set threshold above or below
    fillcolor -- any color type that plotly can handle
    layer -- position of shape in plotly fiugre, like "below"
    
    """
    
    if mode == 'above':
        m = df[variable].gt(level)
    
    if mode == 'below':
        m = df[variable].lt(level)
        
    df1 = df[m].groupby((~m).cumsum())['time'].agg(['first','last'])

    for index, row in df1.iterrows():
        #print(row['first'], row['last'])
        fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0=row['first'],
            y0=0,
            x1=row['last'],
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(100,100,100,0.2)",
            layer=layer
        )
    return(fig)


In [25]:
del fig
agg_func = {'TOTALMWh':'sum','EHF_flag':'max', 'EHF_val':'max','TOTALCLEARED':'sum'}
fuel_grp = df.groupby(['fuel_source_primary', 'time']).aggregate(agg_func).reset_index()
fuel_grp['FRACCLEAR'] = fuel_grp['TOTALMWh']/fuel_grp['TOTALCLEARED']

fig = plot_agg_group(fuel_grp, group_col='fuel_source_primary', title='Hourly energy generation by technology',highlight=True,y='TOTALMWh')

highLights(
        df=fuel_grp,             # Only this group's data
        fig=fig,
        variable='EHF_flag',  # Column to check
        level=0,              # Threshold
        mode='above',         # or 'below'
        fillcolor='rgba(255,0,0,0.1)',  # Semi-transparent red
        layer='below'
    )

# fig.add_trace(go.Scatter(
#             x=demand.index,
#             y=demand['TOTALDEMAND'],
#             mode='lines',
#             line=dict(
#                 color='blue',
#                 width=1,
#                 dash='dot'
#                 ),
#             name=str('State Demand (MWh)')
#         ))

fig.show()

In [17]:
minmax = df.groupby(['fuel_source_primary','time']).agg(
               min_MWh = pd.NamedAgg(column='cap_norm', aggfunc='min'),
               avg_MWh = pd.NamedAgg(column='cap_norm', aggfunc='mean'),
               max_MWh = pd.NamedAgg(column='cap_norm', aggfunc='max')
               ).reset_index()

In [18]:
def plot_shade_line(agg_df):
    """
    Plots time series lines with shaded min/max bounds per category.

    Parameters:
    - agg_df: DataFrame with columns:
        - 'time': datetime
        - 'fuel_source_primary': category name
        - 'avg_MWh': mean value at each time and category
        - 'min_MWh': min value at each time and category
        - 'max_MWh': max value at each time and category

    Returns:
    - Plotly Figure object
    """

    fig = go.Figure()
    categories = agg_df['fuel_source_primary'].unique()

    rgba_colors = {
        category: color for category, color in zip(
            categories,
            [
                'rgba(31, 119, 180, 0.2)',  # blue
                'rgba(255, 127, 14, 0.2)',  # orange
                'rgba(44, 160, 44, 0.2)',   # green
                'rgba(214, 39, 40, 0.2)',   # red
                'rgba(148, 103, 189, 0.2)'  # purple
            ]
        )
    }

    line_colors = {
        category: color.replace('0.2', '1.0')
        for category, color in rgba_colors.items()
    }

    for category in categories:
        cat_df = agg_df[agg_df['fuel_source_primary'] == category]

        # Shaded area between min and max
        fig.add_trace(go.Scatter(
            x=pd.concat([cat_df['time'], cat_df['time'][::-1]]),
            y=pd.concat([cat_df['max_MWh'], cat_df['min_MWh'][::-1]]),
            fill='toself',
            fillcolor=rgba_colors[category],
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False,
            name=f'{category} Range'
        ))

        # Mean line (avg_MWh)
        fig.add_trace(go.Scatter(
            x=cat_df['time'],
            y=cat_df['avg_MWh'],
            mode='lines',
            name=f'{category} Mean',
            line=dict(width=2, color=line_colors[category])
        ))

    fig.update_layout(
        title='Time series with % of registered capacity and min-max shading for wind farms',
        xaxis_title='time',
        yaxis_title='% of Reg Cap',
        template='plotly_white',
        hovermode='x unified'
    )

    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(count=1,
                         label="1m",
                         step="month",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )

    fig.add_shape(
        type="rect",
        xref="x",
        yref="paper",
        x0='2019-01-24',
        y0=0,
        x1='2019-01-26',
        y1=1,
        line=dict(color="rgba(0,0,0,0)", width=3),
        fillcolor="rgba(255,0,0,0.2)",
        layer='below')

    fig.add_shape(
            type="rect",
            xref="x",
            yref="paper",
            x0='2019-01-13',
            y0=0,
            x1='2019-02-10',
            y1=1,
            line=dict(color="rgba(0,0,0,0)", width=3),
            fillcolor="rgba(0,0,0,0.1)",
            layer='below')

    return fig


ftypes = minmax.groupby('fuel_source_primary',as_index = False)
fdf = ftypes.get_group('Wind')

fig = plot_shade_line(fdf)

In [19]:
fig.show()